# Bulgarian Audio Transcription

Transcribe Bulgarian audio files using multiple ASR approaches.

## Lessons Learned from Initial Tests

| Model | Result | Issue |
|---|---|---|
| small | All `...` | Completely failed on low-bitrate BG audio |
| medium | Mixed BG/English garbage | Hallucinations: "Episode 6", "limitation", "Victor chapel" |
| turbo | Mixed languages | Hallucinations: "Juda", "regardless", "voulez" |

**Root causes:** 64 kbps conversational audio + smaller models = hallucinations.

**Fixes applied below:**
1. Use `large-v3` (much better for non-English)
2. Set `condition_on_previous_text=False` (prevents hallucination cascades)
3. Use **Groq API** (free, runs large-v3 on fast hardware)
4. Use HuggingFace fine-tuned Bulgarian model as alternative

## 1. Setup & Imports

In [1]:
import json
import os
import subprocess
from pathlib import Path

import whisper
import torch

# Fix: use system ffmpeg if conda ffmpeg has broken libopenh264
try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
except (subprocess.CalledProcessError, OSError):
    # Prepend /usr/bin so system ffmpeg is found first
    os.environ["PATH"] = "/usr/bin:" + os.environ.get("PATH", "")
    print("Using system ffmpeg (conda ffmpeg has broken libopenh264)")

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"ffmpeg: {subprocess.run(['which', 'ffmpeg'], capture_output=True, text=True).stdout.strip()}")

Using system ffmpeg (conda ffmpeg has broken libopenh264)
Torch version: 2.8.0+cu128
CUDA available: False
Device: cpu
ffmpeg: /usr/bin/ffmpeg


In [ ]:
# Paths
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# List available files
audio_files = sorted(DATA_DIR.glob("*.mp3")) + sorted(DATA_DIR.glob("*.mp4"))
for f in audio_files:
    print(f"{f.name:60s} {f.stat().st_size / 1024 / 1024:8.1f} MB")

mp3_files = sorted(DATA_DIR.glob("*.mp3"))

## 2. Approach A: Groq API (FREE, Recommended)

Sign up at https://console.groq.com (free, no credit card needed), get an API key.
Runs Whisper large-v3 on Groq's LPU hardware — fast and accurate.

```
pip install groq
export GROQ_API_KEY='gsk_...'
```

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

GROQ_KEY = os.environ["GROQ_API_KEY"]
print(f"Groq API key: ...{GROQ_KEY[-4:]}")

### Transcribe with Groq (large-v3, free)

In [ ]:
from groq import Groq

groq_results = {}

if GROQ_KEY:
    client = Groq(api_key=GROQ_KEY)
    
    for audio_path in mp3_files:
        print(f"\nTranscribing: {audio_path.name}")
        with open(audio_path, "rb") as f:
            response = client.audio.transcriptions.create(
                file=(audio_path.name, f.read()),
                model="whisper-large-v3",
                language="bg",
                response_format="verbose_json",
                temperature=0.0,
            )
        groq_results[audio_path.name] = response.model_dump()
        
        # Preview
        print(f"  Segments: {len(response.segments)}")
        for seg in response.segments[:5]:
            print(f"  [{seg.start:.1f}s - {seg.end:.1f}s] {seg.text.strip()}")
        if len(response.segments) > 5:
            print(f"  ... ({len(response.segments) - 5} more)")
else:
    print("Skipping Groq — no API key. Jump to Approach B.")

In [ ]:
# Save Groq results
for name, result in groq_results.items():
    save_transcript(result, name, "groq_large_v3")

## 3. Approach B: Local Whisper large-v3 (free, no API needed)

Best settings for low-quality Bulgarian audio:
- `large-v3` model (much better for non-English than medium/turbo)
- `condition_on_previous_text=False` — prevents hallucination cascades
- `no_speech_threshold=0.6` — stricter silence filtering
- Slow on CPU (~20-30 min for 2 min audio) but works

In [ ]:
import time

MODEL_NAME = "large-v3"

print(f"Loading model: {MODEL_NAME} (this takes ~30s and uses ~6GB RAM)")
model = whisper.load_model(MODEL_NAME)
print(f"Model loaded on: {model.device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

### Transcribe with anti-hallucination settings

In [ ]:
local_results = {}

for audio_path in mp3_files:
    print(f"\nTranscribing: {audio_path.name} ({audio_path.stat().st_size / 1024 / 1024:.1f} MB)")
    t0 = time.time()
    
    result = model.transcribe(
        str(audio_path),
        language="bg",
        task="transcribe",
        verbose=False,
        word_timestamps=True,
        condition_on_previous_text=False,  # KEY: prevents hallucination cascades
        no_speech_threshold=0.6,           # stricter silence detection
        compression_ratio_threshold=2.4,   # reject garbled segments
    )
    
    elapsed = time.time() - t0
    local_results[audio_path.name] = result
    
    print(f"  Time: {elapsed:.0f}s | Segments: {len(result['segments'])}")
    for seg in result["segments"][:5]:
        s, e = seg["start"], seg["end"]
        print(f"  [{s:.1f}s - {e:.1f}s] {seg['text'].strip()}")
    if len(result["segments"]) > 5:
        print(f"  ... ({len(result['segments']) - 5} more)")

# Save
for name, result in local_results.items():
    save_transcript(result, name, "large_v3")

print(f"\nResults saved to: {OUTPUT_DIR}")

## 4. Approach C: HuggingFace Fine-tuned Bulgarian Model

`sam8000/whisper-large-v3-turbo-bulgarian-bulgaria` is fine-tuned specifically
for Bulgarian (9.97% WER on FLEURS). Faster than large-v3 (turbo variant).

In [ ]:
# pip install transformers accelerate
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

BG_MODEL_ID = "sam8000/whisper-large-v3-turbo-bulgarian-bulgaria"

print(f"Loading fine-tuned BG model: {BG_MODEL_ID}")
try:
    bg_processor = AutoProcessor.from_pretrained(BG_MODEL_ID)
    bg_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        BG_MODEL_ID,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    
    bg_pipe = hf_pipeline(
        "automatic-speech-recognition",
        model=bg_model,
        tokenizer=bg_processor.tokenizer,
        feature_extractor=bg_processor.feature_extractor,
        torch_dtype=torch.float32,
        device="cpu",
    )
    print("Model loaded!")
except Exception as e:
    print(f"Failed to load model: {e}")
    print("Try: pip install transformers accelerate")
    bg_pipe = None

In [ ]:
bg_results = {}

if bg_pipe:
    for audio_path in mp3_files:
        print(f"\nTranscribing: {audio_path.name}")
        t0 = time.time()
        
        result = bg_pipe(
            str(audio_path),
            generate_kwargs={"language": "bulgarian", "task": "transcribe"},
            return_timestamps=True,
            chunk_length_s=30,
            batch_size=1,
        )
        
        elapsed = time.time() - t0
        bg_results[audio_path.name] = result
        
        print(f"  Time: {elapsed:.0f}s")
        if "chunks" in result:
            for chunk in result["chunks"][:5]:
                ts = chunk.get("timestamp", (0, 0))
                print(f"  [{ts[0]:.1f}s - {ts[1]:.1f}s] {chunk['text'].strip()}")
        else:
            print(f"  {result['text'][:300]}")
else:
    print("BG model not loaded. Use Approach A or B instead.")

## 5. Compare All Approaches

Side-by-side comparison of transcript quality across methods.

In [ ]:
# Load all saved transcripts for comparison
import glob as globmod

print("All saved transcripts:")
print(f"{'File':<55} {'Size':>8}")
print("-" * 65)
for f in sorted(OUTPUT_DIR.glob("*.txt")):
    print(f"{f.name:<55} {f.stat().st_size / 1024:>7.1f} KB")

# Compare first file across models
test_file = "audio1"
print(f"\n{'='*70}")
print(f" Comparison for: {test_file}")
print(f"{'='*70}")

for txt in sorted(OUTPUT_DIR.glob(f"{test_file}_*.txt")):
    model_tag = txt.stem.replace(f"{test_file}_", "")
    content = txt.read_text(encoding="utf-8")
    lines = content.strip().split("\n")
    print(f"\n--- {model_tag} ({len(lines)} segments) ---")
    for line in lines[:8]:
        print(f"  {line}")
    if len(lines) > 8:
        print(f"  ... ({len(lines) - 8} more)")

## 6. Save & Export Helpers

In [ ]:
def save_transcript(result: dict, audio_name: str, model_name: str):
    """Save transcript as JSON, TXT, and SRT."""
    stem = Path(audio_name).stem
    segments = result.get("segments", [])
    
    # JSON
    json_path = OUTPUT_DIR / f"{stem}_{model_name}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2, default=str)
    
    # TXT
    txt_path = OUTPUT_DIR / f"{stem}_{model_name}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start, end = seg.get("start", 0), seg.get("end", 0)
            m1, s1 = divmod(start, 60)
            m2, s2 = divmod(end, 60)
            f.write(f"[{int(m1):02d}:{s1:05.2f} - {int(m2):02d}:{s2:05.2f}] {seg.get('text', '').strip()}\n")
    
    # SRT
    srt_path = OUTPUT_DIR / f"{stem}_{model_name}.srt"
    with open(srt_path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            h1, r1 = divmod(seg.get("start", 0), 3600); m1, s1 = divmod(r1, 60)
            h2, r2 = divmod(seg.get("end", 0), 3600); m2, s2 = divmod(r2, 60)
            f.write(f"{i}\n")
            f.write(f"{int(h1):02d}:{int(m1):02d}:{int(s1):02d},{int((s1%1)*1000):03d} --> ")
            f.write(f"{int(h2):02d}:{int(m2):02d}:{int(s2):02d},{int((s2%1)*1000):03d}\n")
            f.write(f"{seg.get('text', '').strip()}\n\n")
    
    print(f"Saved: {json_path.name}, {txt_path.name}, {srt_path.name}")

In [ ]:
# List all output files
print("All saved outputs:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name:55s} {f.stat().st_size / 1024:>7.1f} KB")